# Lab 02 - 3 #

## Binary image classification on the DogsVsCats dataset using a ResNet-18 ##

In [3]:
import os
import torch
import torchvision

import torch.nn as nn
import torchvision.models as models

from PIL import Image
# from tqdm import tqdm
from pathlib import Path
from tqdm.notebook import tqdm
from collections import OrderedDict
from torchvision import datasets, transforms
from torchvision.models import ResNet18_Weights
from torchvision.io import decode_image 
from torch.utils.data import Dataset, DataLoader

# NOTE: you can download the dataset here: https://www.kaggle.com/datasets/moazeldsokyx/dogs-vs-cats
# Dataset path.
dataset_path = Path('dataset')
print(f"Dataset path: {dataset_path.resolve()}")

# Hyperparameters.
LR = 1e-4
EPOCH = 5
BATCH_SIZE = 50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Dataset path: C:\Users\marco\Desktop\UNIVERSITA\MAGISTRALE\CV-lab\Lab02\dataset


* Here you have to decide whether to build a dataset class from scratch, or use one provided by the torchvision library. Please have a look to:
- https://pytorch.org/tutorials/beginner/basics/data_tutorial.html (Go to section "Creating a Custom Dataset for yout files")
- https://docs.pytorch.org/vision/main/generated/torchvision.datasets.ImageFolder.html

In [6]:
# Transformations.
data_transform = transforms.Compose([transforms.Resize((224, 224)),
                                     transforms.ToTensor(),
                                     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],)])


train_dataset = datasets.ImageFolder(root=dataset_path / 'train', transform=data_transform)
test_dataset = datasets.ImageFolder(root=dataset_path / 'test', transform=data_transform)
validation_dataset = datasets.ImageFolder(root=dataset_path / 'validation', transform=data_transform)

# DataLoaders.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Verify class mapping
print("Class mapping:", train_dataset.class_to_idx)

Class mapping: {'cats': 0, 'dogs': 1}


* Define a ResNet-18 model, and adapt it to the binary classification task. Recall you have to modify the last layer of the model in order to output a single value.

In [ ]:
# Define the model.
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 1)

model = model.to(DEVICE)

# Define the loss function.
criterion = nn.BCEWithLogitsLoss()

# Define the optimizer.
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

* Write the training loop

In [15]:
losses = []

# Training step.
print(f"Start training on {DEVICE} [...]")
model.train() # Set to training mode

for e in range(EPOCH):
    print(f'EPOCH {e + 1}:')
    e_loss = 0.0

    for i, data in (tepoch := tqdm(enumerate(train_loader), unit="batch", total=len(train_loader))):
        inputs, labels = data
        
        # Move to device and reshape labels to [BATCH_SIZE, 1] as floats
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE).float().unsqueeze(1)
        
        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Update parameters
        optimizer.step()
        
        e_loss += loss.item()
        tepoch.set_postfix(loss=loss.item())
        
    avg_loss = e_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Average Loss for EPOCH {e + 1}: {avg_loss:.4f}")

Start training on cuda [...]
EPOCH 1:


  0%|          | 0/400 [00:00<?, ?batch/s]

Average Loss for EPOCH 1: 0.0591
EPOCH 2:


  0%|          | 0/400 [00:00<?, ?batch/s]

Average Loss for EPOCH 2: 0.0140
EPOCH 3:


...

Average Loss for EPOCH 3: 0.0130
EPOCH 4:


  0%|          | 0/400 [00:00<?, ?batch/s]

Average Loss for EPOCH 4: 0.0074
EPOCH 5:


  0%|          | 0/400 [00:00<?, ?batch/s]

Average Loss for EPOCH 5: 0.0066


* Write the evaluation step

In [16]:
# Evaluation step.
t_loss = 0.0
correct = 0
total = 0

model.eval() # Set model to evaluation mode

with torch.no_grad():
    for i, data in (tepoch := tqdm(enumerate(test_loader), unit="batch", total=len(test_loader))):
        inputs, labels = data
        
        # Format labels 
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE).float().unsqueeze(1)
        
        # Forward pass
        outputs = model(inputs)
        
        # Calculate loss
        loss = criterion(outputs, labels)
        t_loss += loss.item()
        
        # Apply sigmoid to convert logits to probabilities, then round to 0 or 1
        predicted = torch.round(torch.sigmoid(outputs))
        
        # Calculate accuracy
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

avg_test_loss = t_loss / len(test_loader)
accuracy = 100 * correct / total

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {accuracy:.2f}%")

  0%|          | 0/250 [00:00<?, ?batch/s]

Test Loss: 0.0429
Test Accuracy: 98.75%
